# Lab | Web Scraping — *Books to Scrape*

## 1. Context and objective

Publishing companies want to understand the landscape of **highly-rated, reasonably priced books** across genres in order to guide their marketing and publishing decisions.

In this lab, I play the role of a data analyst who collects this information directly from the website [Books to Scrape](https://books.toscrape.com/), a sandbox bookstore built for practising web scraping.

**Goal:** build a reusable function `scrape_books(min_rating, max_price)` that:

- scrapes the catalogue with **`requests`** and **`BeautifulSoup`**,
- keeps only the books whose rating is **at least `min_rating`** and whose price is **at most `max_price`**,
- can optionally restrict the result to **specific genres** through a third, optional parameter `genres`,
- returns a clean **`pandas` DataFrame** with 7 columns: `UPC`, `Title`, `Price (£)`, `Rating`, `Genre`, `Availability`, `Description`.

The function is then run with **`min_rating = 4.0`** and **`max_price = £20`**, and the result is checked and summarised by genre.

## 2. Approach

After inspecting the HTML of the website, I identified two levels of pages:

| Page type | Information available | How I use it |
|---|---|---|
| **Listing pages** (`catalogue/page-1.html` … `page-50.html`) | title, price, star rating, link to the book | loop over all pages and **filter** on rating and price |
| **Book detail pages** | UPC, genre, availability, full description | visited **only for the books that passed the filters** |

Key design choices:

1. **Pagination:** the function keeps requesting `page-1`, `page-2`, … until the server answers with a *404*, so it does not depend on a hard-coded number of pages.
2. **Filter first, then visit:** the rating and the price are already visible on the listing page. Filtering there avoids downloading hundreds of detail pages that would be thrown away.
3. **Modular code:** small helper functions (download, price parsing, detail-page parsing) keep the main function short and easy to test.
4. **Optional genre filter:** the genre only appears on the detail page, so the `genres` filter is applied right after the detail page is parsed. The two required parameters keep the exact signature asked by the lab, and `genres=None` (default) means "all genres".
5. **Good scraping manners:** a short pause (`time.sleep`) between detail-page requests, and a timeout on every request.

## 3. Setup

### 3.1 Libraries

- `requests` downloads the HTML pages,
- `BeautifulSoup` parses the HTML and lets me select elements with CSS selectors,
- `pandas` structures the result as a table,
- `re` (regular expressions) cleans the price text,
- `time` adds a pause between requests,
- `urljoin` builds absolute URLs from the relative links found in the HTML.

In [1]:
import re
import time
from urllib.parse import urljoin

import pandas as pd
import requests
from bs4 import BeautifulSoup

### 3.2 Constants

- `CATALOGUE_URL` is a template: `{}` is replaced by the page number.
- `RATING_MAP` converts the rating written in words in the HTML (for example `<p class="star-rating Four">`) into an integer, which makes the comparison with `min_rating` possible.

In [2]:
BASE_URL = "https://books.toscrape.com/"
CATALOGUE_URL = BASE_URL + "catalogue/page-{}.html"

RATING_MAP = {"One": 1, "Two": 2, "Three": 3, "Four": 4, "Five": 5}

## 4. Helper functions

### 4.1 `get_soup(url)` — download and parse a page

Downloads a page and returns a `BeautifulSoup` object.

- Returns `None` on a **404**: this is the signal that the last catalogue page has been passed.
- `raise_for_status()` raises an error for any other HTTP problem instead of silently continuing with bad data.
- `response.encoding = "utf-8"` forces the correct encoding, otherwise the pound sign appears as `Â£`.

In [3]:
def get_soup(url):
    """Download a page and return a BeautifulSoup object (None if 404)."""
    response = requests.get(url, timeout=10)
    if response.status_code == 404:
        return None
    response.raise_for_status()
    response.encoding = "utf-8"  # avoids 'Â£' in prices
    return BeautifulSoup(response.text, "html.parser")

### 4.2 `parse_price(text)` — clean the price

The price is displayed as text (`"£51.77"`). A regular expression removes every character that is not a digit or a dot, then the result is converted to a `float` so it can be compared with `max_price`.

In [4]:
def parse_price(text):
    """'£51.77' -> 51.77"""
    return float(re.sub(r"[^\d.]", "", text))

### 4.3 `parse_book_page(url)` — extract the details of one book

Visits the page of a single book and extracts the four fields that are not available on the listing page:

- **UPC:** read in the product information table (the `<td>` next to the `<th>UPC</th>`),
- **Genre:** the third element of the breadcrumb (`Home > Books > Genre > Title`),
- **Availability:** normalised to `"In stock"` or `"Out of stock"`,
- **Description:** the paragraph following the `product_description` block. Some books have no description, so the value stays `None` in that case instead of raising an error.

In [5]:
def parse_book_page(url):
    """Extract UPC, genre, availability and description from a book page."""
    soup = get_soup(url)

    upc = soup.find("th", string="UPC").find_next_sibling("td").get_text(strip=True)

    # Breadcrumb: Home > Books > Genre > Title
    genre = soup.select("ul.breadcrumb li")[2].get_text(strip=True)

    availability_text = soup.select_one("p.availability").get_text(strip=True)
    availability = "In stock" if "In stock" in availability_text else "Out of stock"

    desc_header = soup.find("div", id="product_description")
    description = None
    if desc_header:
        description = desc_header.find_next_sibling("p").get_text(strip=True)

    return upc, genre, availability, description

## 5. Main function: `scrape_books(min_rating, max_price)`

This is the function requested by the lab. Its logic, step by step:

1. Loop over the catalogue pages until a page returns `None` (404).
2. For each book on the page, read its **rating** and **price**.
3. **Skip** the book if `rating < min_rating` or `price > max_price`.
4. For the books that pass the filters, build the absolute URL of the book and call `parse_book_page` to get UPC, genre, availability and description. If a `genres` list was given, skip the books whose genre is not in it.
5. Store each book as a dictionary in a list.
6. Convert the list into a **DataFrame** with the exact column names required by the lab.

Since ratings are whole numbers from 1 to 5, `min_rating=4.0` keeps the books rated **4 or 5 stars**.

In [6]:
def scrape_books(min_rating, max_price, genres=None, delay=0.1):
    """
    Scrape Books to Scrape and return a DataFrame of books with
    rating >= min_rating and price <= max_price.
    If `genres` is given (list of genre names), only books of these genres are kept.
    """
    wanted_genres = {g.lower() for g in genres} if genres else None
    books = []
    page = 1

    while True:
        soup = get_soup(CATALOGUE_URL.format(page))
        if soup is None:  # no more pages
            break

        for article in soup.select("article.product_pod"):
            # Rating: <p class="star-rating Three">
            rating_word = article.select_one("p.star-rating")["class"][1]
            rating = RATING_MAP[rating_word]

            price = parse_price(article.select_one("p.price_color").get_text())

            # Filter BEFORE visiting the detail page (much faster)
            if rating < min_rating or price > max_price:
                continue

            link = article.h3.a
            title = link["title"]
            book_url = urljoin(CATALOGUE_URL.format(page), link["href"])

            upc, genre, availability, description = parse_book_page(book_url)
            time.sleep(delay)  # be polite with the server

            # Optional genre filter (the genre is only available on the detail page)
            if wanted_genres and genre.lower() not in wanted_genres:
                continue

            books.append(
                {
                    "UPC": upc,
                    "Title": title,
                    "Price (£)": price,
                    "Rating": rating,
                    "Genre": genre,
                    "Availability": availability,
                    "Description": description,
                }
            )

        page += 1

    return pd.DataFrame(
        books,
        columns=["UPC", "Title", "Price (£)", "Rating", "Genre", "Availability", "Description"],
    )


## 6. Execution

### 6.1 Required run

The function is executed with the parameters requested by the lab: **minimum rating of 4.0** and **maximum price of £20**. The first rows of the DataFrame are displayed below.

In [8]:
df = scrape_books(min_rating=4.0, max_price=20)
df.head()

,UPC,Title,Price (£),Rating,Genre,Availability,Description
0,ce6396b0f23f6ecc,Set Me Free,17.46,5,Young Adult,In stock,Aaron Ledbetter’s future had been planned out ...
1,6258a1f6a6dcfe50,The Four Agreements: A Practical Guide to Pers...,17.66,5,Spirituality,In stock,"In The Four Agreements, don Miguel Ruiz reveal..."
2,6be3beb0793a53e7,Sophie's World,15.94,5,Philosophy,In stock,A page-turning novel that is also an explorati...
3,657fe5ead67a7767,Untitled Collection: Sabbath Poems 2014,14.27,4,Poetry,In stock,"More than thirty-five years ago, when the weat..."
4,51653ef291ab7ddc,This One Summer,19.49,4,Sequential Art,In stock,"Every summer, Rose goes with her mom and dad t..."


**What the first rows show:**

- The function returns **75 books** described by **7 columns**, exactly as required.
- All the books displayed have a rating of 4 or 5 and a price below £20, which is consistent with the filters.
- The genres are already varied (Young Adult, Spirituality, Philosophy, Poetry, Sequential Art…).
- Prices are stored as numbers (not text) and ratings as integers, so the data is **ready for analysis**.

### 6.2 Filtering on specific genres

The lab also mentions filtering on specific genres. This is done with the optional `genres` parameter. Example: among the books rated 4+ and priced up to £20, keep only **Poetry** and **Philosophy**.

In [ ]:
df_genres = scrape_books(min_rating=4.0, max_price=20, genres=["Poetry", "Philosophy"])
df_genres[["Title", "Genre", "Price (£)", "Rating"]]

## 7. Validation and analysis

### 7.1 Sanity checks

Before analysing the data, we check that the DataFrame respects the requested filters and that the data is clean:

- the lowest rating must be at least 4 and the highest price at most £20 (the `assert` statements would raise an error otherwise),
- no book should appear twice (duplicated UPC),
- I count the books without description, since this field is optional on the website.

In [ ]:
print("Shape (rows, columns):", df.shape)
print("Lowest rating:", df["Rating"].min(), "| Highest price (£):", df["Price (£)"].max())

assert df["Rating"].min() >= 4, "A book below the minimum rating slipped through"
assert df["Price (£)"].max() <= 20, "A book above the maximum price slipped through"

print("Duplicated UPC:", df["UPC"].duplicated().sum())
print("Books without description:", df["Description"].isna().sum())

### 7.2 Summary by genre

To answer the publishers' question, I group the books by genre and compute, for each genre, the **number of books**, the **average price** and the **average rating**. The table is sorted by number of books.

In [ ]:
genre_summary = (
    df.groupby("Genre")
      .agg(books=("UPC", "count"),
           avg_price=("Price (£)", "mean"),
           avg_rating=("Rating", "mean"))
      .round(2)
      .sort_values("books", ascending=False)
)
genre_summary

The chart below shows the 10 genres with the most highly-rated books under £20.

In [ ]:
import matplotlib.pyplot as plt

top_genres = genre_summary["books"].head(10).sort_values()
top_genres.plot(kind="barh", figsize=(8, 5),
                title="Top 10 genres — books rated 4+ and priced up to £20")
plt.xlabel("Number of books")
plt.ylabel("")
plt.tight_layout()
plt.show()

## 8. Conclusion, limitations and possible improvements

**What I built:** a versatile scraper that can be re-run with any combination of `min_rating` and `max_price` (for example `scrape_books(5, 10)` for the cheapest top-rated books, or `scrape_books(4, 20, genres=["Poetry"])` for one genre) and that returns a tidy DataFrame. The checks confirm that the filters are correctly applied, and the summary by genre shows where highly-rated, affordable books are concentrated and how their prices compare across genres: exactly the kind of information a publisher can use to guide marketing and publishing decisions.

**Limitations:**

- The scraper is designed for the structure of this website: if the HTML changes, the selectors will need to be updated.
- Requests are sent one by one, so a run with loose filters takes longer (many detail pages to visit), and the genre filter still needs to visit the detail page of every book that passes the rating and price filters.

**Possible improvements:**

- Use the website's **category pages** to reach a genre directly, instead of visiting the detail page of books from other genres.
- Use a `requests.Session` and error handling with retries to make the scraper more robust.
- Export the result to CSV for use in other tools.